# Poly(A) site resolution: distance to AATAAA in organoid (standard 10x 3') vs mouse (extended Read 1)

**Author:** Andre Lopes Martins Macedo
**Date:** 2026-10-07
**Purpose:** Measure how sharply poly(A) sites are resolved in the human cortical organoid data (sites from Read 2 ends) compared with the mouse data (extended Read 1 reading the cleavage site), using the distance of every molecule's assigned site to the nearest canonical AATAAA hexamer.

## 1. Setup

Imports, paths and constants. The PAS-distance helpers are imported from `<repo>/src/peak_calling.py`, the exact module used to compute Fig. 1B (`pc.pas_distance`, hamming = 0), so the distance definition is identical to the paper:

- exact AATAAA match (hamming 0), searched on the strand-oriented 3'UTR sequence (overlapping matches allowed);
- signed distance = site position minus hexamer end, both in 0-based transcript-oriented coordinates of the same annotated 3'UTR (the hexamer end is the 0-based exclusive end, as returned by `_find_pas`), positive = site downstream of the hexamer;
- nearest hexamer by absolute distance (`pc._calculate_umi_distance_to_pas`);
- UTRs without a hexamer are excluded; every molecule (UMI) counts, i.e. each site is weighted by its total UMI count over all cells.

The utrcalling package version of `_calculate_umi_distance_to_pas` is NOT used: it multiplies minus-strand distances by -1 (although the sequence is already reverse-complemented) and prefers any hexamer in [10, 30] over the nearest one, so it would not reproduce Fig. 1B. Outputs go to a new folder; the helper `new_file` refuses to overwrite anything. No randomness is involved; a seed is set for completeness.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pysam

sys.path.append(f"{REPO_ROOT}/src/")
import peak_calling as pc  # module used for Fig. 1B

SEED = 0
np.random.seed(SEED)

# Inputs
ORGANOID_H5AD = Path(
    f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/"
    "merged_bw12_above10cts_apa_named_annotated_fix_pt.h5ad"
)
MOUSE_FINAL_H5AD = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/"
    "UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad"
)
MOUSE_FIG1B_H5AD = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/utrcalling/"
    "UTR_count_table_bw12_above5perc_apa_named.h5ad"
)
FIG1B_CACHE = Path(
    f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling/"
    "DirMult/pas_distances_hamming0.npy"
)
GENOMES = Path(f"{DATA_ROOT}/common_files/genomes")
ANNOT = Path(f"{DATA_ROOT}/common_files/annotations")
HUMAN_FASTA = GENOMES / "Homo_sapiens.GRCh38.dna.primary_assembly.fa"
MOUSE_FASTA = GENOMES / "Mus_musculus.GRCm38.dna.primary_assembly.fa"
HUMAN_GTF = ANNOT / "Homo_sapiens.GRCh38.104.gtf"
MOUSE_GTF = ANNOT / "Mus_musculus.GRCm38.102.gtf"

# Outputs (new folder; never overwrite)
OUT_DIR = Path(f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling/pas_distance")
OUT_DIR.mkdir(parents=False, exist_ok=True)


def new_file(name: str) -> Path:
    """Return a path in OUT_DIR, refusing to overwrite an existing file."""
    path = OUT_DIR / name
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite {path}")
    return path


WINDOW = (10, 30)
FIG1B_PCT = 49.1
DATASET_LABELS = {
    "mouse_final": "Mouse, extended Read 1 (final matrix)",
    "organoid": "Human organoid, standard 10x 3' (Read 2)",
}
SHORT_LABELS = {"mouse_final": "Mouse (extended Read 1)", "organoid": "Organoid (Read 2)"}
COLORS = {"mouse_final": "#4393C3", "organoid": "#D6604D"}
print(f"Outputs -> {OUT_DIR}")

## 2. Helper functions: gene-level coordinates, hexamers, weighted summaries

Both final matrices are GENE-LEVEL utrcalling objects: all annotated 3'UTR segments of a gene (bedtools-merged, strand-aware, so segments of one gene are disjoint) are concatenated on one axis, and `var.utr_size` is the position on that axis. `var.start` / `var.end` list the segment starts and ends (1-based, GTF-like) joined by `|`, in ascending genomic order. In `process_mapped_reads.merge_utrs_per_gene` / `_compute_gene_offsets`, segments are ordered in transcription direction (ascending for +, descending for -) and segment k gets the offset `sum over previous segments of (end - start)`. Within a segment, utrcalling's `utr_size = |read_pos - start_d|` with `start_d = start - 1` (+) or `end - 1` (-), so position `u` along the segment is 0 at its 5'-most base in transcript orientation, which is exactly index 0 of the reverse-complemented (for -) segment sequence used by `_find_pas`.

Conversion used here: a site at axis position `a` is assigned to the segment with the largest offset <= `a`, `u = a - offset`, genomic 1-based position = `start + u` (+) or `end - u` (-). Because a segment spans `end - start + 2` axis values but the next offset only advances by `end - start`, the first two axis values of every non-first segment are shared with the last two of the previous segment; such sites (within 1 nt of a segment junction) are flagged as `junction_ambiguous` and reported.

In [ ]:
HEXAMER_RE = re.compile(r"(?=AATAAA)")  # overlapping exact matches, as in pc._find_pas
COMPLEMENT = str.maketrans("ACGTNacgtn", "TGCANtgcan")


def reverse_complement(seq: str) -> str:
    return seq.translate(COMPLEMENT)[::-1]


def build_segment_table(var: pd.DataFrame) -> pd.DataFrame:
    """One row per (gene, annotated 3'UTR segment), in transcription order, with gene-axis offsets."""
    coord_cols = ["chrom", "start", "end", "strand"]
    n_unique = var.groupby("utr_name", observed=True)[coord_cols].nunique()
    assert (n_unique == 1).all().all(), "segment coordinates differ within a gene"
    genes = var.drop_duplicates("utr_name")
    rows: list[tuple] = []
    for gene, chrom, start, end, strand in zip(
        genes["utr_name"].astype(str),
        genes["chrom"].astype(str),
        genes["start"].astype(str),
        genes["end"].astype(str),
        genes["strand"].astype(str),
    ):
        starts = [int(x) for x in start.split("|")]
        ends = [int(x) for x in end.split("|")]
        assert len(starts) == len(ends), gene
        segs = list(zip(starts, ends))
        assert all(s <= e for s, e in segs), gene
        assert all(s2 > e1 for (_, e1), (s2, _) in zip(segs, segs[1:])), gene  # disjoint, ascending
        if strand == "-":
            segs = segs[::-1]
        offset = 0
        for rank, (s, e) in enumerate(segs):
            rows.append((gene, f"{gene}#{rank}", rank, len(segs), chrom, s, e, strand, offset))
            offset += e - s
    return pd.DataFrame(
        rows,
        columns=["gene", "segment_id", "segment_rank", "n_segments", "chrom",
                 "seg_start", "seg_end", "strand", "offset"],
    )


def assign_sites(var: pd.DataFrame, segments: pd.DataFrame, umis: np.ndarray) -> pd.DataFrame:
    """Place every site (var row) on its 3'UTR segment and genomic coordinate."""
    sites = pd.DataFrame({
        "site_idx": np.arange(var.shape[0]),
        "site_id": var.index.astype(str),
        "gene": var["utr_name"].astype(str).to_numpy(),
        "axis_pos": var["utr_size"].astype(np.int64).to_numpy(),
        "umis": umis.astype(np.int64),
    })
    merged = sites.merge(segments, on="gene", how="left")
    merged = merged[merged["offset"] <= merged["axis_pos"]]
    merged = merged.sort_values(["site_idx", "offset", "segment_rank"])
    merged = merged.drop_duplicates("site_idx", keep="last").sort_values("site_idx")
    assert len(merged) == len(sites), "some sites could not be placed on a segment"
    merged["pos_in_segment"] = merged["axis_pos"] - merged["offset"]
    seg_len = merged["seg_end"] - merged["seg_start"] + 1
    merged["beyond_segment_end"] = merged["pos_in_segment"] > seg_len
    merged["junction_ambiguous"] = (merged["segment_rank"] > 0) & (merged["pos_in_segment"] <= 1)
    merged["genomic_pos"] = np.where(
        merged["strand"] == "+",
        merged["seg_start"] + merged["pos_in_segment"],
        merged["seg_end"] - merged["pos_in_segment"],
    )
    return merged.reset_index(drop=True)


def hexamer_ends(fasta: pysam.FastaFile, chrom: str, start: int, end: int, strand: str) -> list[int]:
    """0-based exclusive ends of AATAAA in the transcript-oriented segment sequence."""
    seq = fasta.fetch(chrom, start - 1, end)
    if strand == "-":
        seq = reverse_complement(seq)
    return [m.start() + 6 for m in HEXAMER_RE.finditer(seq)]


def pas_distances_per_site(sites: pd.DataFrame, fasta_path: Path) -> pd.DataFrame:
    """Nearest-hexamer signed distance per site (NaN if its segment has no AATAAA)."""
    sites = sites.copy()
    sites["n_hexamers"] = 0
    sites["pas_distance"] = np.nan
    n_lower = 0
    with pysam.FastaFile(str(fasta_path)) as fasta:
        missing = set(sites["chrom"]) - set(fasta.references)
        assert not missing, f"contigs missing from FASTA: {missing}"
        for _, idx in sites.groupby("segment_id", sort=False).indices.items():
            first = sites.iloc[idx[0]]
            seq = fasta.fetch(first["chrom"], int(first["seg_start"]) - 1, int(first["seg_end"]))
            n_lower += sum(c.islower() for c in seq)
            ends = hexamer_ends(fasta, first["chrom"], int(first["seg_start"]),
                                int(first["seg_end"]), first["strand"])
            sites.iloc[idx, sites.columns.get_loc("n_hexamers")] = len(ends)
            if not ends:
                continue
            u = sites["pos_in_segment"].to_numpy()[idx]
            d = pc._calculate_umi_distance_to_pas(umi_utr_sizes=u, pas_end=ends)
            sites.iloc[idx, sites.columns.get_loc("pas_distance")] = np.asarray(d, dtype=float)
    print(f"  lower-case bases in fetched segments: {n_lower}")
    return sites


def weighted_order_stat(values: np.ndarray, weights: np.ndarray, k: int) -> float:
    """k-th (0-based) element of the values array expanded by integer weights."""
    order = np.argsort(values, kind="stable")
    cw = np.cumsum(weights[order])
    return float(values[order][np.searchsorted(cw, k, side="right")])


def weighted_median(values: np.ndarray, weights: np.ndarray) -> float:
    """Median of the expanded array (identical to np.median on np.repeat(values, weights))."""
    n = int(weights.sum())
    if n % 2 == 1:
        return weighted_order_stat(values, weights, n // 2)
    return 0.5 * (weighted_order_stat(values, weights, n // 2 - 1)
                  + weighted_order_stat(values, weights, n // 2))


def summarise(sites: pd.DataFrame, label: str) -> dict[str, float | int | str]:
    """Molecule-weighted PAS-distance summary over sites in hexamer-containing UTRs."""
    s = sites[sites["pas_distance"].notna() & (sites["umis"] > 0)]
    d = s["pas_distance"].to_numpy()
    w = s["umis"].to_numpy()
    n = int(w.sum())
    return {
        "dataset": label,
        "n_utr_segments_with_hexamer": int(s["segment_id"].nunique()),
        "n_sites": int(len(s)),
        "n_molecules": n,
        "pct_molecules_10_30nt": 100 * w[(d >= 10) & (d <= 30)].sum() / n,
        "pct_molecules_9_29nt_lastA_convention": 100 * w[(d >= 9) & (d <= 29)].sum() / n,
        "median_abs_distance_nt": weighted_median(np.abs(d), w),
        "median_signed_distance_nt": weighted_median(d, w),
        "pct_molecules_within_50nt": 100 * w[np.abs(d) <= 50].sum() / n,
        "pct_molecules_0_100nt": 100 * w[(d >= 0) & (d <= 100)].sum() / n,
        "pct_molecules_upstream_of_hexamer": 100 * w[d < 0].sum() / n,
        "pct_molecules_junction_ambiguous": 100 * w[s["junction_ambiguous"].to_numpy()].sum() / n,
    }


def column_umis(adata: ad.AnnData) -> np.ndarray:
    return np.asarray(adata.X.sum(axis=0)).ravel().round().astype(np.int64)

## 3. Validation 1: reproduce Fig. 1B with this code

Before touching the new data, run the same code on the 2024 UTR-level mouse object that Fig. 1B was computed on (one segment per `utr_name`, so the offset logic is trivially 0) and compare with the cached Fig. 1B distance array. If the code is equivalent, the molecule-weighted distance distribution must be identical to the cache (49.1% in [10, 30]).

In [ ]:
adata_fig1b = ad.read_h5ad(MOUSE_FIG1B_H5AD)
seg_fig1b = build_segment_table(adata_fig1b.var)
print(f"Fig. 1B object: {adata_fig1b.shape}, segments per utr_name: {seg_fig1b['n_segments'].max()}")
sites_fig1b = assign_sites(adata_fig1b.var, seg_fig1b, column_umis(adata_fig1b))
sites_fig1b = pas_distances_per_site(sites_fig1b, MOUSE_FASTA)
del adata_fig1b

cache = np.load(FIG1B_CACHE)
s = sites_fig1b[sites_fig1b["pas_distance"].notna()]
mine = s.groupby("pas_distance")["umis"].sum()
mine = mine[mine > 0]
vals, counts = np.unique(cache, return_counts=True)
cached = pd.Series(counts, index=vals.astype(float))
identical = mine.index.equals(cached.index) and np.array_equal(mine.to_numpy(), cached.to_numpy())
fig1b_check = summarise(sites_fig1b, "mouse_fig1b_2024_object")
print(f"cache: {len(cache):,} molecules, {100 * ((cache >= 10) & (cache <= 30)).mean():.2f}% in [10, 30]")
print(f"this code: {fig1b_check['n_molecules']:,} molecules, {fig1b_check['pct_molecules_10_30nt']:.2f}% in [10, 30]")
print(f"distance distributions identical: {identical}")
assert identical, "code does not reproduce Fig. 1B"

## 4. Load the final matrices and build the gene-level segment tables

Load the final mouse gene-level matrix and the organoid gene-level matrix, sum UMIs per site over all cells (X holds raw integer UMI counts), and expand the `|`-joined segment coordinates into one row per annotated 3'UTR segment with its gene-axis offset.

In [ ]:
DATASETS = {
    "mouse_final": {"h5ad": MOUSE_FINAL_H5AD, "fasta": MOUSE_FASTA, "gtf": MOUSE_GTF},
    "organoid": {"h5ad": ORGANOID_H5AD, "fasta": HUMAN_FASTA, "gtf": HUMAN_GTF},
}
segments: dict[str, pd.DataFrame] = {}
sites: dict[str, pd.DataFrame] = {}
for name, cfg in DATASETS.items():
    adata = ad.read_h5ad(cfg["h5ad"])
    x_data = adata.X.data if hasattr(adata.X, "data") else np.asarray(adata.X)
    assert np.all(x_data == np.round(x_data)), "X is not integer counts"
    umis = column_umis(adata)
    segments[name] = build_segment_table(adata.var)
    sites[name] = assign_sites(adata.var, segments[name], umis)
    seg = segments[name]
    print(
        f"{name}: {adata.n_obs:,} cells x {adata.n_vars:,} sites, {umis.sum():,} molecules; "
        f"{seg['gene'].nunique():,} genes, {len(seg):,} segments "
        f"({(seg.groupby('gene')['n_segments'].first() > 1).mean():.1%} of genes with >1 segment); "
        f"sites beyond segment end: {int(sites[name]['beyond_segment_end'].sum())}"
    )
    del adata

## 5. Validation 2: segment offsets match utrcalling's own `_compute_gene_offsets`

The offsets above are re-derived from the `var` coordinates. To make sure they are the offsets utrcalling actually applied when building the gene-level axis, every gene's segments are passed to `utrcalling.core.process_mapped_reads._compute_gene_offsets` in the `utrcalling` conda env (the package is not importable in `downstr`), and the two offset sets are compared. The utrcalling code that built both matrices (2026-03 and 2026-04 runs) has the same offset logic as the current source (last change 2026-02-25).

In [ ]:
UTRCALLING_SNIPPET = """
import json, sys
import pandas as pd
from utrcalling.core.process_mapped_reads import _compute_gene_offsets
seg = pd.read_csv(sys.stdin, dtype={"gene": str, "utr_name": str, "strand": str})
out = {}
for _, grp in seg.groupby("gene", sort=False):
    out.update({k: int(v) for k, v in _compute_gene_offsets(grp[["start", "end", "strand", "utr_name"]]).items()})
print(json.dumps(out))
"""
conda_exe = os.environ.get("CONDA_EXE", "conda")
utrcalling_python = subprocess.run(
    [conda_exe, "run", "-n", "utrcalling", "python", "-c", "import sys; print(sys.executable)"],
    capture_output=True, text=True, check=True,
).stdout.strip().splitlines()[-1]
print(f"utrcalling interpreter: {utrcalling_python}")

offset_checks = []
for name, seg in segments.items():
    payload = seg.rename(columns={"seg_start": "start", "seg_end": "end", "segment_id": "utr_name"})[
        ["gene", "utr_name", "start", "end", "strand"]
    ].to_csv(index=False)
    res = subprocess.run([utrcalling_python, "-c", UTRCALLING_SNIPPET], input=payload,
                         capture_output=True, text=True, check=True)
    ref = json.loads(res.stdout.strip().splitlines()[-1])
    ref_offsets = seg["segment_id"].map(ref)
    n_match = int((ref_offsets == seg["offset"]).sum())
    offset_checks.append({"dataset": name, "n_segments": len(seg), "n_offsets_matching_utrcalling": n_match})
    print(f"{name}: {n_match:,} / {len(seg):,} segment offsets identical to utrcalling")
    assert n_match == len(seg)

## 6. Distance of every molecule to the nearest AATAAA

For each segment that carries at least one site, fetch its genomic sequence (GRCm38 / GRCh38 primary assembly, the references used by the utrcalling runs), orient it, find all exact AATAAA matches, and compute the nearest-hexamer signed distance with the Fig. 1B function `pc._calculate_umi_distance_to_pas`. Each site is then weighted by its UMI count.

In [ ]:
for name, cfg in DATASETS.items():
    print(name)
    sites[name] = pas_distances_per_site(sites[name], cfg["fasta"])
    s = sites[name]
    frac = s.loc[s["pas_distance"].notna(), "umis"].sum() / s["umis"].sum()
    print(f"  molecules in hexamer-containing UTR segments: {frac:.1%}")

## 7. Validation 3: genomic placement of sites vs annotated transcript 3' ends

The PAS distance only uses the position within the segment, but a wrong segment assignment or offset would scramble both. As an independent check, the converted genomic position of every site is compared with the nearest annotated transcript 3' end (GTF `transcript` end for +, start for -; same chromosome and strand). For the mouse data (cleavage-site resolution) sites in non-first segments should be as close to annotated ends as sites in first segments, and the AATAAA signal should be present on both strands and in both segment classes. The organoid comparison shows where Read 2-based sites sit relative to annotated ends.

In [ ]:
def load_annotated_three_prime_ends(gtf: Path) -> dict[tuple[str, str], np.ndarray]:
    gtf_df = pd.read_csv(
        gtf, sep="\t", comment="#", header=None, usecols=[0, 2, 3, 4, 6],
        names=["chrom", "feature", "start", "end", "strand"],
        dtype={"chrom": str, "feature": str, "start": np.int64, "end": np.int64, "strand": str},
    )
    tx = gtf_df[gtf_df["feature"] == "transcript"]
    tx = tx.assign(end3=np.where(tx["strand"] == "+", tx["end"], tx["start"]))
    return {key: np.unique(g["end3"].to_numpy()) for key, g in tx.groupby(["chrom", "strand"])}


def distance_to_annotated_end(s: pd.DataFrame, ends: dict[tuple[str, str], np.ndarray]) -> np.ndarray:
    """Signed distance (transcript orientation) from each site to the nearest annotated 3' end."""
    out = np.full(len(s), np.nan)
    for (chrom, strand), idx in s.groupby(["chrom", "strand"]).indices.items():
        ref = ends.get((chrom, strand))
        if ref is None:
            continue
        pos = s["genomic_pos"].to_numpy()[idx]
        j = np.clip(np.searchsorted(ref, pos), 1, len(ref) - 1)
        left, right = ref[j - 1], ref[j]
        nearest = np.where(np.abs(pos - left) <= np.abs(pos - right), left, right)
        out[idx] = (pos - nearest) if strand == "+" else (nearest - pos)
    return out


validation_rows = []
for name, cfg in DATASETS.items():
    ends = load_annotated_three_prime_ends(cfg["gtf"])
    s = sites[name]
    s["dist_to_annotated_3p_end"] = distance_to_annotated_end(s, ends)
    s["segment_class"] = np.where(s["segment_rank"] == 0, "first segment", "later segments")
    for (strand, seg_class), g in s.groupby(["strand", "segment_class"]):
        w = g["umis"].to_numpy()
        dend = g["dist_to_annotated_3p_end"].to_numpy()
        hx = g["pas_distance"].notna().to_numpy()
        dpas = g["pas_distance"].to_numpy()
        validation_rows.append({
            "dataset": name, "strand": strand, "segment_class": seg_class,
            "n_sites": len(g), "n_molecules": int(w.sum()),
            "pct_molecules_within_25nt_of_annotated_3p_end": 100 * w[np.abs(dend) <= 25].sum() / w.sum(),
            "median_signed_dist_to_annotated_3p_end_nt": weighted_median(dend[~np.isnan(dend)], w[~np.isnan(dend)]),
            "pct_molecules_10_30nt_from_AATAAA": 100 * w[hx & (dpas >= 10) & (dpas <= 30)].sum() / w[hx].sum(),
        })
    del ends
validation = pd.DataFrame(validation_rows)
validation.round(2)

## 8. Example genes

A few highly expressed multi-segment genes per dataset whose dominant site lies in a non-first segment, showing the axis position, the assigned segment, the converted genomic coordinate, and its distance to the nearest annotated 3' end and to the nearest AATAAA.

In [ ]:
cols = ["gene", "site_id", "strand", "chrom", "segment_rank", "n_segments", "seg_start", "seg_end",
        "axis_pos", "pos_in_segment", "genomic_pos", "umis", "dist_to_annotated_3p_end", "pas_distance"]
for name, s in sites.items():
    top = s.sort_values("umis", ascending=False).drop_duplicates("gene")
    ex = pd.concat([
        top[(top["segment_rank"] > 0) & (top["strand"] == strand)].head(3) for strand in ["+", "-"]
    ])
    print(name)
    print(ex[cols].to_string(index=False))

## 9. Summary statistics

Molecule-weighted statistics over all molecules whose site lies in a hexamer-containing 3'UTR segment (denominator as in Fig. 1B: includes negative distances and distances > 100 nt). Also reported: median absolute distance, fraction within +-50 nt, and, as a 1-nt convention check, the fraction in [9, 29] (equivalent to [10, 30] if the distance is measured from the last A of the hexamer instead of its exclusive end).

In [ ]:
summary = pd.DataFrame([fig1b_check] + [summarise(sites[name], name) for name in DATASETS])
summary.insert(1, "description", summary["dataset"].map({
    "mouse_fig1b_2024_object": "Fig. 1B input (2024 UTR-level mouse object), recomputed",
    "mouse_final": DATASET_LABELS["mouse_final"],
    "organoid": DATASET_LABELS["organoid"],
}))
summary.to_csv(new_file("pas_distance_summary.tsv"), sep="\t", index=False)
validation.to_csv(new_file("validation_by_strand_and_segment.tsv"), sep="\t", index=False)
pd.DataFrame(offset_checks).to_csv(new_file("validation_offsets_vs_utrcalling.tsv"), sep="\t", index=False)
summary.set_index("dataset").T

## 10. Figure: distance histograms (Fig. 1B style)

Left and middle: molecule-weighted histograms of distances in [0, 100] nt (1-nt bins, dashed lines at 10 and 30 nt, as in Fig. 1B); the percentage is computed over all molecules in hexamer-containing UTRs. Right: the full picture from -400 to +200 nt as percentage of molecules per 5-nt bin, which shows where Read 2-derived organoid sites sit relative to the hexamer.

In [ ]:
plt.rcParams.update({"font.size": 8, "axes.spines.top": False, "axes.spines.right": False})
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), gridspec_kw={"width_ratios": [1, 1, 1.4]})
for ax, name in zip(axes[:2], DATASETS):
    s = sites[name][sites[name]["pas_distance"].notna()]
    row = summary.set_index("dataset").loc[name]
    ax.hist(s["pas_distance"], bins=100, range=(0, 100), weights=s["umis"],
            color=COLORS[name], alpha=0.85, edgecolor="none")
    for x in WINDOW:
        ax.axvline(x, color="black", ls="--", lw=0.8, alpha=0.7)
    ax.text(0.97, 0.95, f"{row['pct_molecules_10_30nt']:.1f}% in [10, 30] nt",
            transform=ax.transAxes, ha="right", va="top",
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8, edgecolor="gray"))
    ax.set_xlim(0, 100)
    ax.set_title(DATASET_LABELS[name], fontsize=8)
    ax.set_xlabel("Distance to AATAAA (nt)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v / 1e6:g}"))
    ax.set_ylabel("Molecules (millions)")

ax = axes[2]
bins = np.arange(-400, 205, 5)
for name in DATASETS:
    s = sites[name][sites[name]["pas_distance"].notna()]
    h, _ = np.histogram(s["pas_distance"], bins=bins, weights=s["umis"])
    ax.stairs(100 * h / s["umis"].sum(), bins, color=COLORS[name], lw=1.2, label=SHORT_LABELS[name])
for x in WINDOW:
    ax.axvline(x, color="black", ls="--", lw=0.8, alpha=0.7)
ax.set_xlabel("Distance to nearest AATAAA (nt)")
ax.set_ylabel("% of molecules per 5-nt bin")
ax.legend(frameon=False, fontsize=7, loc="upper left")
fig.tight_layout()
fig.savefig(new_file("pas_distance_histograms.png"), dpi=300)
fig.savefig(new_file("pas_distance_histograms.pdf"))
plt.show()

## 11. Save per-site tables

Per-site results for both datasets (segment assignment, genomic coordinate, UMI count, distance to the nearest AATAAA and to the nearest annotated 3' end) for re-use.

In [ ]:
keep = ["site_id", "gene", "chrom", "strand", "segment_rank", "n_segments", "seg_start", "seg_end",
        "offset", "axis_pos", "pos_in_segment", "genomic_pos", "junction_ambiguous", "umis",
        "n_hexamers", "pas_distance", "dist_to_annotated_3p_end"]
for name, s in sites.items():
    path = new_file(f"pas_distance_per_site_{name}.tsv.gz")
    s[keep].to_csv(path, sep="\t", index=False)
    print(path)
sorted(p.name for p in OUT_DIR.iterdir())

## 12. Summary

**Results** (molecule-weighted; denominator = all molecules whose site lies in an AATAAA-containing annotated 3'UTR segment, including negative distances and distances > 100 nt, as in Fig. 1B):

| Dataset | Molecules | % in [10, 30] nt | Median abs. distance (nt) | % within +-50 nt |
|---|---|---|---|---|
| Fig. 1B input (2024 mouse UTR-level object), recomputed | 9,540,761 | 49.1 | 20 | 58.6 |
| Mouse, final gene-level matrix (extended Read 1) | 9,911,647 | 50.9 | 19 | 60.5 |
| Human organoid, gene-level (standard 10x 3', Read 2) | 218,714,935 | 2.9 | 59 | 36.6 |

- The mouse final matrix gives 50.9% in [10, 30] nt, consistent with the 49.1% shown in Fig. 1B (computed on the older 2024 object; Fig. 1B is unchanged).
- In the organoid data the window holds only 2.9% of molecules, about the same as the mirror window [-30, -10] (3.2%), so there is no enrichment downstream of the hexamer. 80% of organoid molecules sit upstream of their nearest AATAAA (median signed distance -56 nt; median distance to the nearest annotated transcript 3' end -49 nt). This matches Read 2 ends falling tens of nt upstream of the cleavage site. Organoid sites therefore resolve poly(A) sites to roughly +-50-100 nt, not to the cleavage site.

**Validation**
- The code reproduces the Fig. 1B distance distribution exactly (identical molecule-level distances, 9,540,761 molecules, 49.12%).
- Gene-level segment offsets are identical to utrcalling's `_compute_gene_offsets` for all 29,757 mouse and 66,443 organoid segments.
- In the mouse data, sites in non-first segments lie at annotated 3' ends (median -2 nt) and show the AATAAA signal (53-54% in [10, 30] nt) on both strands, which confirms the axis-to-genome conversion.

**Caveats**
- The distance convention follows Fig. 1B: hexamer end = 0-based exclusive end. Measuring from the last A instead shifts the window by 1 nt (mouse final 51.9%, organoid 4.1% in [9, 29]).
- Only exact AATAAA is used. Segments without a hexamer are excluded (13-17% of molecules).
- The nearest hexamer is chosen by absolute distance within the same merged annotated 3'UTR segment, so hexamers in other segments or outside the annotation are ignored.
- Sites are KDE peak centres (bandwidth 12), not single reads.
- 0.28% (mouse) and 0.04% (organoid) of molecules lie within 1 nt of a segment junction, where the gene axis is ambiguous by construction.

**Outputs** (in `$M26_DATA_ROOT/3utr/organoids/knoblich/utrcalling/pas_distance/`):
- `pas_distance_summary.tsv`: the statistics above for the three datasets
- `pas_distance_histograms.png` / `.pdf`: Fig. 1B-style 0-100 nt histograms plus the -400..+200 nt overlay
- `validation_by_strand_and_segment.tsv`, `validation_offsets_vs_utrcalling.tsv`: conversion checks
- `pas_distance_per_site_mouse_final.tsv.gz`, `pas_distance_per_site_organoid.tsv.gz`: per-site segment, genomic position, UMIs and distances